# Data Cleaning

We will prepare the data for analysis

In [ ]:
# Import necessary libraries
import os
import duckdb
import pandas as pd
import glob


## 1. Loading Datasets

### Merging all datasets into on csv for Rwanda using DuckDB

In [ ]:
# Define paths
input_pattern = "data/raw/wfp_food_prices_global_*.csv"
output_path = "data/raw/rwanda_wfp_prices_2020_2026.csv"

os.makedirs(os.path.dirname(output_path), exist_ok=True)

# The global WFP files are large and are not committed to the repository. If the
# Rwanda extract already exists, reuse it; only re-run the extraction when the
# source files are actually present. This keeps the notebook runnable from a
# clone that carries the extract but not the multi-GB global inputs.
source_files = glob.glob(input_pattern)

if os.path.exists(output_path) and not source_files:
    n = sum(1 for _ in open(output_path)) - 1
    print(f"Reusing existing Rwanda extract: {output_path}")
    print(f"Total Rows: {n:,}")
elif not source_files:
    raise FileNotFoundError(
        f"Neither the global WFP files ({input_pattern}) nor an existing extract "
        f"({output_path}) were found. Download the WFP global price CSVs into data/raw/."
    )
else:
    print("Starting streaming extraction via DuckDB engine...")
    con = duckdb.connect()
    con.execute(f"""
        COPY (
            SELECT *
            FROM read_csv_auto('{input_pattern}')
            WHERE TRIM(countryiso3) = 'RWA'
        ) TO '{output_path}' (HEADER, DELIMITER ',');
    """)
    total_rows = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{output_path}')").fetchone()[0]
    print(f"Master file saved to: {output_path}")
    print(f"Total Rows Saved: {total_rows:,}")


Starting streaming extraction via DuckDB engine...
Master file saved to: data/raw/rwanda_wfp_prices_2020_2026.csv
Total Rows Saved: 26,182


### Loading merged dataset to dataframe

In [3]:
# Read the Rwanda CSV file into a DataFrame to verify the results
rwanda_df = pd.read_csv(output_path)
rwanda_df.head(5)

,countryiso3,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,RWA,2020-01-15,Eastern Province,Gatsibo,Bubare,2373,-1.6,30.25,cereals and tubers,Maize flour,76,KG,actual,Retail,RWF,733.33,0.78
1,RWA,2020-01-15,Eastern Province,Gatsibo,Bubare,2373,-1.6,30.25,cereals and tubers,Rice (imported),64,KG,actual,Retail,RWF,933.33,1.00
2,RWA,2020-01-15,Eastern Province,Gatsibo,Bubare,2373,-1.6,30.25,cereals and tubers,Rice (local),71,KG,actual,Retail,RWF,800.00,0.85
3,RWA,2020-01-15,Eastern Province,Gatsibo,Bubare,2373,-1.6,30.25,miscellaneous food,Salt,185,KG,actual,Retail,RWF,366.67,0.39
4,RWA,2020-01-15,Eastern Province,Gatsibo,Bubare,2373,-1.6,30.25,oil and fats,Oil,137,L,actual,Retail,RWF,1700.00,1.81


## 2. Dataset Inspection

In [4]:
# Basic info
print("Shape:", rwanda_df.shape)
print("\nColumn names:", rwanda_df.columns.tolist())
print("\nNumber of duplicate rows:", rwanda_df.duplicated().sum())
print("\nData types:\n", rwanda_df.dtypes)

Shape: (26182, 17)

Column names: ['countryiso3', 'date', 'admin1', 'admin2', 'market', 'market_id', 'latitude', 'longitude', 'category', 'commodity', 'commodity_id', 'unit', 'priceflag', 'pricetype', 'currency', 'price', 'usdprice']

Number of duplicate rows: 0

Data types:
 countryiso3         str
date                str
admin1              str
admin2              str
market              str
market_id         int64
latitude        float64
longitude       float64
category            str
commodity           str
commodity_id      int64
unit                str
priceflag           str
pricetype           str
currency            str
price           float64
usdprice        float64
dtype: object


In [5]:
print("\nMissing values per column:\n", rwanda_df.isnull().sum())


Missing values per column:
 countryiso3     0
date            0
admin1          0
admin2          0
market          0
market_id       0
latitude        0
longitude       0
category        0
commodity       0
commodity_id    0
unit            0
priceflag       0
pricetype       0
currency        0
price           0
usdprice        0
dtype: int64


*Note: we can see that there is not missing values or duplicates*

In [6]:
# Convert date and create features
rwanda_df['date'] = pd.to_datetime(rwanda_df['date'])
rwanda_df['year'] = rwanda_df['date'].dt.year
rwanda_df['month'] = rwanda_df['date'].dt.month
rwanda_df['quarter'] = rwanda_df['date'].dt.quarter
rwanda_df['day_of_week'] = rwanda_df['date'].dt.dayofweek

In [7]:
# 1. Get description of specific numerical price columns
price_summary = rwanda_df[['price', 'usdprice']].describe()
print("--- Price Summary Metrics ---")
print(price_summary)

--- Price Summary Metrics ---
              price      usdprice
count  26182.000000  26182.000000
mean    1057.880689      0.893653
std     1000.493635      0.809515
min       50.000000      0.053000
25%      400.000000      0.350000
50%      715.450000      0.600000
75%     1300.000000      1.070000
max     8017.780000      5.500000


### Rename columns

In [8]:
# Rename columns for clarity and consistency
column_rename = {
    'countryiso3': 'country',
    'admin1': 'province',
    'admin2': 'district',
    'market': 'market_name',
    'market_id': 'market_id',
    'latitude': 'lat',
    'longitude': 'lon',
    'category': 'food_category',
    'commodity': 'commodity_name',
    'commodity_id': 'commodity_id',
    'unit': 'unit',
    'priceflag': 'price_flag',
    'pricetype': 'price_type',
    'currency': 'currency',
    'price': 'price_rwf',
    'usdprice': 'price_usd'
}

rwanda_df.rename(columns=column_rename, inplace=True)
rwanda_df.columns

Index(['country', 'date', 'province', 'district', 'market_name', 'market_id',
       'lat', 'lon', 'food_category', 'commodity_name', 'commodity_id', 'unit',
       'price_flag', 'price_type', 'currency', 'price_rwf', 'price_usd',
       'year', 'month', 'quarter', 'day_of_week'],
      dtype='str')

In [9]:
# 1. Filter to consistent units (KG)
# For your regression/classification tasks, filter to only KG commodities so all prices are per kilogram. 
# You can keep the L data for clustering if you want, but for supervised learning, standardise on one unit.
rwanda_df = rwanda_df[rwanda_df['unit'] == 'KG'].copy()

# 2. Filter to Retail prices (consumer-level)
rwanda_df = rwanda_df[rwanda_df['price_type'] == 'Retail'].copy()

# 3. Drop redundant ID columns
rwanda_df = rwanda_df.drop(columns=['market_id', 'commodity_id'], errors='ignore')

# 4. Filter to 'actual' price flags
# Keep only 'actual' to only have real observed prices, not estimated or forecasted ones.
rwanda_df = rwanda_df[rwanda_df['price_flag'] == 'actual'].copy()

# 5. Create a new column 'commodity_group'
def group_commodity(name):
    if 'Potato' in name:
        return 'Potatoes'
    elif 'Rice' in name:
        return 'Rice'
    elif 'Maize' in name and 'flour' in name:
        return 'Maize flour'
    elif 'Maize' in name:
        return 'Maize'
    elif 'Cassava' in name and 'flour' in name:
        return 'Cassava flour'
    elif 'Cassava' in name:
        return 'Cassava'
    else:
        return name  # Keeps Sugar, Beans, Meat, Salt, etc. as they are

rwanda_df['commodity_group'] = rwanda_df['commodity_name'].apply(group_commodity)

print("Grouped distribution:")
print(rwanda_df['commodity_group'].value_counts())

# 6. Check the new shape
print(f"Final shape after deep cleaning: {rwanda_df.shape}")

Grouped distribution:
commodity_group
Potatoes         2376
Cassava          1732
Sugar            1355
Cassava flour    1252
Fish (dry)       1238
Sorghum          1223
Bananas          1148
Rice             1130
Cabbage          1092
Maize flour       700
Meat (beef)       681
Salt              419
Beans (dry)       406
Maize             376
Name: count, dtype: int64
Final shape after deep cleaning: (15128, 20)


In [10]:
rwanda_df.head(5)

,country,date,province,district,market_name,lat,lon,food_category,commodity_name,unit,price_flag,price_type,currency,price_rwf,price_usd,year,month,quarter,day_of_week,commodity_group
0,RWA,2020-01-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,cereals and tubers,Maize flour,KG,actual,Retail,RWF,733.33,0.78,2020,1,1,2,Maize flour
1,RWA,2020-01-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,cereals and tubers,Rice (imported),KG,actual,Retail,RWF,933.33,1.00,2020,1,1,2,Rice
2,RWA,2020-01-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,cereals and tubers,Rice (local),KG,actual,Retail,RWF,800.00,0.85,2020,1,1,2,Rice
3,RWA,2020-01-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,miscellaneous food,Salt,KG,actual,Retail,RWF,366.67,0.39,2020,1,1,2,Salt
5,RWA,2020-01-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,pulses and nuts,Beans (dry),KG,actual,Retail,RWF,460.00,0.49,2020,1,1,2,Beans (dry)


In [11]:
rwanda_df["province"].value_counts()

province
Eastern Province     6677
Southern Province    5083
Western Province     2586
Northern Province     682
Kigali City           100
Name: count, dtype: int64

In [12]:
rwanda_df.to_csv("data/cleaned/rwanda_wfp_prices_clean.csv", index=False)
print("Cleaning complete. New shape:", rwanda_df.shape)
print("New columns:", rwanda_df.columns.tolist())

Cleaning complete. New shape: (15128, 20)
New columns: ['country', 'date', 'province', 'district', 'market_name', 'lat', 'lon', 'food_category', 'commodity_name', 'unit', 'price_flag', 'price_type', 'currency', 'price_rwf', 'price_usd', 'year', 'month', 'quarter', 'day_of_week', 'commodity_group']


## Merge Price Data with Temperature Data

In [13]:

# 1. Load the cleaned price data
price_df = rwanda_df.copy()  # Use the cleaned DataFrame directly instead of reading from CSV

# 2. Load the temperature data
temp_df = pd.read_csv("data/raw/rwanda_temperature_2020_2026.csv")

In [14]:
temp_df['date'] = pd.to_datetime(temp_df['date'])
temp_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12185 entries, 0 to 12184
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              12185 non-null  datetime64[us]
 1   location          12185 non-null  str           
 2   lat               12185 non-null  float64       
 3   lon               12185 non-null  float64       
 4   temp_celsius      12185 non-null  float64       
 5   precipitation_mm  12185 non-null  float64       
 6   soil_moisture     12185 non-null  float64       
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 666.5 KB


In [15]:
# 3. Rename 'location' column to 'province' for matching
temp_df = temp_df.rename(columns={'location': 'province'})

# 4. Merge on date and province (LEFT JOIN to keep all price rows)
merged_df = price_df.merge(temp_df, on=['date', 'province'], how='left')

merged_df.head(5)

,country,date,province,district,market_name,lat_x,lon_x,food_category,commodity_name,unit,...,year,month,quarter,day_of_week,commodity_group,lat_y,lon_y,temp_celsius,precipitation_mm,soil_moisture
0,RWA,2020-01-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,cereals and tubers,Maize flour,KG,...,2020,1,1,2,Maize flour,-1.95,30.45,20.9,0.51,0.82
1,RWA,2020-01-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,cereals and tubers,Rice (imported),KG,...,2020,1,1,2,Rice,-1.95,30.45,20.9,0.51,0.82
2,RWA,2020-01-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,cereals and tubers,Rice (local),KG,...,2020,1,1,2,Rice,-1.95,30.45,20.9,0.51,0.82
3,RWA,2020-01-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,miscellaneous food,Salt,KG,...,2020,1,1,2,Salt,-1.95,30.45,20.9,0.51,0.82
4,RWA,2020-01-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,pulses and nuts,Beans (dry),KG,...,2020,1,1,2,Beans (dry),-1.95,30.45,20.9,0.51,0.82


In [16]:

# Rename columns to avoid confusion and drop redundant lat/lon columns
merged_df = merged_df.rename(columns={'lat_x': 'lat', 'lon_x': 'lon'}).drop(columns=['lat_y', 'lon_y'])


In [17]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15128 entries, 0 to 15127
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   country           15128 non-null  str           
 1   date              15128 non-null  datetime64[us]
 2   province          15128 non-null  str           
 3   district          15128 non-null  str           
 4   market_name       15128 non-null  str           
 5   lat               15128 non-null  float64       
 6   lon               15128 non-null  float64       
 7   food_category     15128 non-null  str           
 8   commodity_name    15128 non-null  str           
 9   unit              15128 non-null  str           
 10  price_flag        15128 non-null  str           
 11  price_type        15128 non-null  str           
 12  currency          15128 non-null  str           
 13  price_rwf         15128 non-null  float64       
 14  price_usd         15128 non-null 

In [18]:
# 5. Check for missing temperature (should be 0 if province names match exactly)
missing_temp = merged_df['temp_celsius'].isnull().sum()
print(f"Rows with missing temperature: {missing_temp} ({missing_temp/len(merged_df)*100:.2f}%)")
if missing_temp > 0:
    print("Provinces with missing temperature:", merged_df[merged_df['temp_celsius'].isnull()]['province'].unique())

Rows with missing temperature: 0 (0.00%)


In [19]:
df = merged_df.copy()

print(f"Original shape: {df.shape}")

# Drop all old, broken, and duplicated weather columns to start fresh
cols_to_drop = [
    "temp_anomaly", "temp_lag_1", "temp_lag_2", "temp_lag_3", "temp_rolling_3mo",
    "monthly_avg_temp", "monthly_avg_temp_x", "monthly_avg_temp_y",
    "precip_anomaly", "precip_lag_1", "precip_lag_2", "precip_lag_3", "precip_rolling_3mo",
    "soil_moisture_lag_1", "monthly_avg_precip",
]
df_cleaned_base = df.drop(
    columns=[col for col in cols_to_drop if col in df.columns]
)

# Create an isolated timeline with exactly ONE row per province per date.
# Now carries precipitation_mm and soil_moisture alongside temp_celsius, so all
# three climate variables get the same leakage-safe lag/rolling treatment.
CLIMATE_RAW_COLS = ["temp_celsius", "precipitation_mm", "soil_moisture"]

weather_timeline = (
    df[["province", "year", "month", "date"] + CLIMATE_RAW_COLS]
    .drop_duplicates()
    .sort_values(["province", "date"])
)

# Monthly climatological baseline and anomaly (temperature and precipitation only;
# soil moisture is already a smoothed/integrated measure, so an "anomaly" on top of
# it would mostly just be noise).
for var, out_col in [("temp_celsius", "temp_anomaly"), ("precipitation_mm", "precip_anomaly")]:
    baseline_col = f"monthly_avg_{var.split('_')[0]}"
    baseline = (
        weather_timeline.groupby(["province", "month"])[var]
        .mean()
        .reset_index()
        .rename(columns={var: baseline_col})
    )
    weather_timeline = weather_timeline.merge(baseline, on=["province", "month"], how="left")
    weather_timeline[out_col] = weather_timeline[var] - weather_timeline[baseline_col]

# Chronological lags: temperature and precipitation get 1-3 month lags (both are
# directly tied to recent growing conditions); soil moisture gets a single 1-month
# lag, since it already integrates recent rainfall history and additional lags would
# be highly collinear with each other.
for lag in [1, 2, 3]:
    weather_timeline[f"temp_lag_{lag}"] = weather_timeline.groupby("province")["temp_celsius"].shift(lag)
    weather_timeline[f"precip_lag_{lag}"] = weather_timeline.groupby("province")["precipitation_mm"].shift(lag)

weather_timeline["soil_moisture_lag_1"] = weather_timeline.groupby("province")["soil_moisture"].shift(1)

# Rolling 3-month averages (temperature and precipitation)
weather_timeline["temp_rolling_3mo"] = weather_timeline.groupby("province")["temp_celsius"].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)
weather_timeline["precip_rolling_3mo"] = weather_timeline.groupby("province")["precipitation_mm"].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

# Merge the engineered climate features back to the main food price dataset.
# Drop the raw CLIMATE_RAW_COLS here since they already exist in df_cleaned_base
# from the original merge — we only need the newly engineered columns from
# weather_timeline, not duplicate copies of the raw values.
final_df = df_cleaned_base.merge(
    weather_timeline.drop(columns=CLIMATE_RAW_COLS),
    on=["province", "year", "month", "date"],
    how="left",
)

print(f"Fixed dataset shape: {final_df.shape}")


Original shape: (15128, 23)
Fixed dataset shape: (15128, 36)


**Known simplification worth disclosing in the paper's Methods/Limitations:** `monthly_baseline` (used for `temp_anomaly`, and now also `precip_anomaly`) is computed as the average value for each `(province, month)` pair across the *entire* 2020-2026 span  meaning a January 2020 row's "seasonal normal" is partly informed by January values from 2021-2026, which hadn't occurred yet. 

This mirrors how meteorologists compute climate normals in practice, so it's a defensible modeling choice, but it is technically a mild form of look-ahead leakage relative to strict forecasting validity. An expanding-window version (using only past years) would be more rigorous if time permits  flagging this explicitly is preferable to leaving it undocumented.

In [20]:
final_df = final_df.sort_values(['province', 'date'])

# List of weather columns to fix — now includes precipitation and soil moisture
# lag/anomaly/rolling features alongside the original temperature ones.
weather_cols = [
    "temp_anomaly", "temp_lag_1", "temp_lag_2", "temp_lag_3", "temp_rolling_3mo",
    "precip_anomaly", "precip_lag_1", "precip_lag_2", "precip_lag_3", "precip_rolling_3mo",
    "soil_moisture_lag_1",
]

# Forward-fill only: a row is allowed to reuse the most recent *past* weather
# reading if one is momentarily missing. We deliberately do NOT back-fill —
# bfill would copy a FUTURE month's weather value backward into an earlier row,
# which is look-ahead leakage, and could never happen in a real forecasting
# deployment.
final_df[weather_cols] = final_df.groupby('province')[weather_cols].ffill()

n_before = len(final_df)
missing_weather_mask = final_df[weather_cols].isna().any(axis=1)
print(f"Rows with no valid weather history yet (start of each province's series): "
      f"{missing_weather_mask.sum()} ({missing_weather_mask.sum()/n_before*100:.2f}% of {n_before:,} rows)")


Rows with no valid weather history yet (start of each province's series): 1190 (7.87% of 15,128 rows)


In [21]:
# Drop rows that still lack real weather history (the start of each province's
# series) rather than fabricating values for them.
final_df = final_df[~missing_weather_mask].copy()
print(f"Dropped {missing_weather_mask.sum()} rows lacking full weather history.")
print(f"Remaining shape: {final_df.shape}")

# Sanity check: no NaNs should remain in the weather columns now
assert final_df[weather_cols].isna().sum().sum() == 0
print("Confirmed: 0 missing values remain in weather columns.")


Dropped 1190 rows lacking full weather history.
Remaining shape: (13938, 36)
Confirmed: 0 missing values remain in weather columns.


In [22]:
final_df.head(5)

,country,date,province,district,market_name,lat,lon,food_category,commodity_name,unit,...,precip_anomaly,temp_lag_1,precip_lag_1,temp_lag_2,precip_lag_2,temp_lag_3,precip_lag_3,soil_moisture_lag_1,temp_rolling_3mo,precip_rolling_3mo
1181,RWA,2020-04-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,cereals and tubers,Cassava,KG,...,-1.265,21.46,4.03,21.02,0.5,20.9,0.51,0.87,21.03,1.883333
1182,RWA,2020-04-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,cereals and tubers,Cassava flour,KG,...,-1.265,21.46,4.03,21.02,0.5,20.9,0.51,0.87,21.03,1.883333
1183,RWA,2020-04-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,cereals and tubers,Maize,KG,...,-1.265,21.46,4.03,21.02,0.5,20.9,0.51,0.87,21.03,1.883333
1184,RWA,2020-04-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,cereals and tubers,Maize flour,KG,...,-1.265,21.46,4.03,21.02,0.5,20.9,0.51,0.87,21.03,1.883333
1185,RWA,2020-04-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,cereals and tubers,Potatoes,KG,...,-1.265,21.46,4.03,21.02,0.5,20.9,0.51,0.87,21.03,1.883333


In [ ]:
# 7. Create the classification target: price direction (increase = 1, decrease/equal = 0)
#
# IMPORTANT: group by commodity_name, not commodity_group. "Rice (imported)" and
# "Rice (local)" are different products at different price levels grouping them
# under the shared label "Rice" would mix two unrelated price series together when
# computing a month-over-month lag, corrupting both price_prev_month and the
# resulting price_direction label.
GROUP_KEYS = ['province', 'market_name', 'commodity_name']

final_df = final_df.sort_values(GROUP_KEYS + ['date'])
final_df['price_prev_month'] = final_df.groupby(GROUP_KEYS)['price_rwf'].shift(1)

n_before = len(final_df)
missing_prev = final_df['price_prev_month'].isna()
print(f"Rows with no previous-month price yet (first observation per series): "
      f"{missing_prev.sum()} ({missing_prev.sum()/n_before*100:.2f}% of {n_before:,} rows)")

# Drop them rather than back-filling. A "previous month" price that is actually a
# LATER month's price copied backward is not something a real forecasting system
# would ever have access to and it would corrupt the price_direction label below,
# since a row's target must be computed from that row's own genuine history.
final_df = final_df[~missing_prev].copy()

# Compute the classification target only now, on real historical prices
# never on a NaN or a back-filled stand-in.
final_df['price_direction'] = (final_df['price_rwf'] > final_df['price_prev_month']).astype(int)

print(f"Remaining shape after dropping rows without real history: {final_df.shape}")
print("\nClass balance for price_direction:")
print(final_df['price_direction'].value_counts(normalize=True))


Rows with no previous-month price yet (first observation per series): 613 (4.40% of 13,938 rows)
Remaining shape after dropping rows without real history: (13325, 38)

Class balance for price_direction:
price_direction
0    0.601276
1    0.398724
Name: proportion, dtype: float64


In [24]:
final_df.head(5)

,country,date,province,district,market_name,lat,lon,food_category,commodity_name,unit,...,precip_lag_1,temp_lag_2,precip_lag_2,temp_lag_3,precip_lag_3,soil_moisture_lag_1,temp_rolling_3mo,precip_rolling_3mo,price_prev_month,price_direction
1597,RWA,2020-05-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,vegetables and fruits,Bananas,KG,...,1.12,21.46,4.03,21.02,0.50,0.88,20.926667,2.906667,200.0,0
3043,RWA,2020-11-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,vegetables and fruits,Bananas,KG,...,0.80,21.72,0.39,20.88,0.25,0.74,21.196667,3.220000,200.0,1
3261,RWA,2020-12-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,vegetables and fruits,Bananas,KG,...,8.47,21.17,0.80,21.72,0.39,0.79,20.656667,4.043333,220.0,1
3463,RWA,2021-01-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,vegetables and fruits,Bananas,KG,...,2.86,20.70,8.47,21.17,0.80,0.83,20.270000,6.116667,230.0,0
3701,RWA,2021-03-15,Eastern Province,Gatsibo,Bubare,-1.6,30.25,vegetables and fruits,Bananas,KG,...,7.02,20.10,2.86,20.70,8.47,0.86,20.273333,3.550000,230.0,0


In [ ]:
# Sanity check: every remaining row now has a genuine (non-fabricated)
# previous-month price and complete weather history  no back-filled or
# leaked values remain anywhere in the modeling features.
assert final_df['price_prev_month'].isna().sum() == 0
assert final_df[weather_cols].isna().sum().sum() == 0
print("No missing or leaked values remain in the price-lag or weather features.")


No missing or leaked values remain in the price-lag or weather features.


*Note: the previous version of this notebook re-filled `price_prev_month` here using `ffill().bfill()` after the fact. That step has been removed  it silently overwrote `price_prev_month` with values (including future ones, via `bfill`) that `price_direction` above had already been computed without seeing, leaving the feature and the label inconsistent with each other. 

Missing-history rows are now dropped once, upstream, before the target is computed  see the two cells above.*

## 9. Merging Macroeconomic Data: Exchange Rate and CPI

Two national-level (not province-specific) time series are added here:
USD/RWF exchange rate and Consumer Price Index (general and food). Both are
merged on `date` alone — every price row for a given month receives the same
national-level value, which is the correct behavior for an economy-wide
covariate (it is not a leakage concern; it's simply a shared feature across
all rows sharing that date).

In [26]:
# --- Exchange rate ---
# Forex rates are observed in real time (no publication lag), so no shift is
# needed here — the rate on a given date genuinely was known on that date.
exchange_df = pd.read_csv("data/raw/rwanda_exchange_2020_2026.csv", parse_dates=["date"])

n_before = len(final_df)
final_df = final_df.merge(exchange_df, on="date", how="left")

missing_fx = final_df["usd_rwf_rate"].isna().sum()
print(f"Rows with missing exchange rate after merge: {missing_fx} ({missing_fx/n_before*100:.2f}%)")


Rows with missing exchange rate after merge: 0 (0.00%)


### CPI: correcting for real-world publication lag

Unlike exchange rates, government CPI figures are **not available in real time** national statistics offices typically publish a given month's CPI several weeks into the following month. If we merged a price row dated `2026-05-15` directly against the CPI row also dated `2026-05-15`, we would be handing the model April's actual, not-yet-published CPI value under May's label the same category of look-ahead leakage already fixed twice elsewhere in this notebook, just relocated to a new data source.

**Disclosed assumption:** we do not have NISR's exact publication calendar, so we assume a conservative one-month lag a price row dated month *M* is only allowed to see the CPI value originally reported for month *M-1*. This is implemented by shifting each CPI row's "availability date" forward by one month before merging, rather than shifting the price data. If NISR's actual lag differs from one month, this is the assumption to revisit first.

In [27]:
# --- CPI (with a 1-month availability lag applied) ---
cpi_df = pd.read_csv("data/raw/rwanda_cpi_2020_2026.csv", parse_dates=["date"])

cpi_shifted = cpi_df.copy()
cpi_shifted["date"] = cpi_shifted["date"] + pd.DateOffset(months=1)
# e.g. April's CPI (reported at 2026-04-15) becomes usable starting 2026-05-15

n_before = len(final_df)
final_df = final_df.merge(cpi_shifted, on="date", how="left")

missing_cpi = final_df["general_cpi"].isna().sum()
print(f"Rows with missing CPI (after 1-month lag shift) after merge: {missing_cpi} ({missing_cpi/n_before*100:.2f}%)")


Rows with missing CPI (after 1-month lag shift) after merge: 0 (0.00%)


In [28]:
# Before dropping anything, measure what each macro column would COST us in rows.
# Dropping rows is the right call for a feature we actually use; it is a waste of
# training data for a feature we don't.
ALL_MACRO_COLS = [
    "usd_rwf_rate", "usd_rwf_mom_pct",
    "general_cpi", "food_cpi", "cpi_mom_pct", "food_cpi_mom_pct", "cpi_yoy_pct",
]

print("Rows that would be lost by requiring each macro column to be non-null:")
for col in ALL_MACRO_COLS:
    n_missing = final_df[col].isna().sum()
    print(f"  {col:20s} {n_missing:6,} rows ({n_missing/len(final_df)*100:5.2f}%)")


Rows that would be lost by requiring each macro column to be non-null:
  usd_rwf_rate              0 rows ( 0.00%)
  usd_rwf_mom_pct           0 rows ( 0.00%)
  general_cpi               0 rows ( 0.00%)
  food_cpi                  0 rows ( 0.00%)
  cpi_mom_pct               0 rows ( 0.00%)
  food_cpi_mom_pct          0 rows ( 0.00%)
  cpi_yoy_pct           2,025 rows (15.20%)


**A data-cost decision, made explicitly rather than by accident.**

`cpi_yoy_pct` is a *year-over-year* change, so it is undefined until 12 months of CPI history exist. Requiring it to be non-null silently deletes roughly the first year of the price data about 15% of all rows, and the earliest part of the series, which is exactly the part a chronological train/test split relies on for training.

That cost is only worth paying if the feature earns it. It does not: the supervised notebook's final feature set uses `food_cpi` and `food_cpi_mom_pct` (month-over-month inflation) and does **not** use `cpi_yoy_pct`, which is largely redundant with them for month-to-month price movement. We therefore drop the `cpi_yoy_pct` *column* and keep the *rows*, rather than the reverse.

This is a deliberate, reversible choice, not a silent one set `KEEP_CPI_YOY = True` below to restore the column and accept the row loss.

In [29]:
KEEP_CPI_YOY = False  # See the note above before changing this.

if KEEP_CPI_YOY:
    required_macro_cols = ALL_MACRO_COLS
else:
    final_df = final_df.drop(columns=["cpi_yoy_pct"])
    required_macro_cols = [c for c in ALL_MACRO_COLS if c != "cpi_yoy_pct"]
    print("Dropped the 'cpi_yoy_pct' COLUMN to preserve ~1 year of training rows.\n")

# Now drop any rows still missing a macro feature we actually intend to use,
# rather than fabricating a value — consistent with how missing weather/price
# history was handled earlier in this notebook.
missing_macro_mask = final_df[required_macro_cols].isna().any(axis=1)

n_before = len(final_df)
final_df = final_df[~missing_macro_mask].copy()
print(f"Dropped {missing_macro_mask.sum()} rows with missing macro data.")
print(f"Remaining: {final_df.shape}")

assert final_df[required_macro_cols].isna().sum().sum() == 0
print("No missing values remain in the macroeconomic features.")


Dropped the 'cpi_yoy_pct' COLUMN to preserve ~1 year of training rows.

Dropped 0 rows with missing macro data.
Remaining: (13325, 44)
No missing values remain in the macroeconomic features.


In [30]:
# 8. Save the enriched dataset
final_df.to_csv("data/cleaned/rwanda_food_prices_with_temperature.csv", index=False)
print(f"✅ Enriched dataset saved with shape: {final_df.shape}")
print("Columns:", final_df.columns.tolist())

✅ Enriched dataset saved with shape: (13325, 44)
Columns: ['country', 'date', 'province', 'district', 'market_name', 'lat', 'lon', 'food_category', 'commodity_name', 'unit', 'price_flag', 'price_type', 'currency', 'price_rwf', 'price_usd', 'year', 'month', 'quarter', 'day_of_week', 'commodity_group', 'temp_celsius', 'precipitation_mm', 'soil_moisture', 'monthly_avg_temp', 'temp_anomaly', 'monthly_avg_precipitation', 'precip_anomaly', 'temp_lag_1', 'precip_lag_1', 'temp_lag_2', 'precip_lag_2', 'temp_lag_3', 'precip_lag_3', 'soil_moisture_lag_1', 'temp_rolling_3mo', 'precip_rolling_3mo', 'price_prev_month', 'price_direction', 'usd_rwf_rate', 'usd_rwf_mom_pct', 'general_cpi', 'food_cpi', 'cpi_mom_pct', 'food_cpi_mom_pct']
